# Fine-tuning Laya for LeetCode difficulty

`laya_experiments.ipynb` showed two things:

* Laya's **zero-shot answer** barely knows what LeetCode difficulty is (macro-F1 ≈ 0.30, 85% "Medium").
* Its **encoder** does: a plain logistic regression on its embeddings reaches ≈ 0.48.

So the knowledge is in the encoder, but the decision head was never taught to turn it into *this* answer.
Fine-tuning does exactly that: we show the head (and the top of the encoder) our labelled training problems.
This is also what the Laya model card recommends ("a fast base to specialise").

**What gets trained.** Same input as the zero-shot `difficulty` question, same three `[MASK]` markers;
only some weights change:

```
[CLS] choice question: How difficult ...? [SEP] [MASK] easy: ... [MASK] medium: ... [MASK] hard: ... [SEP] <problem> [SEP]
        │
        ▼
 ModernBERT-large, 28 layers
   layers  1 … 24        frozen   (general language knowledge, keep it)
   layers 25 … 28        trained  (TRAIN_LAYERS = 4)
        │
        ▼
 Laya decision head (2 small transformer layers) + scorer   trained
        │
        ▼
 one logit per [MASK] → softmax → P(Easy), P(Medium), P(Hard)
```

Freezing the bottom keeps memory and time low (no gradients are stored for frozen layers) and makes
overfitting 1,600 training problems less likely.

**Protocol.** Same split as every other model: the shared test set, and a validation part carved out of the
training part (`split_indices(y, with_val=True)`). Validation picks the epoch (early stopping), the
calibration and the Easy/Hard shift. The test set is used once, at the end.

**Re-running.** The best weights are saved to `trained_models/laya_finetuned.pt` together with the settings
that produced them. On a re-run, if that file exists, `RETRAIN = False` and the settings haven't changed,
training is skipped and the weights are loaded.

In [ ]:
import json
import math
import time

import torch                     # before set_seed(), so torch gets seeded too
import laya
from laya.common import QTYPES, build_sequence, collate_items

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.optimize import minimize
from sklearn.metrics import confusion_matrix, f1_score, log_loss
from tqdm.auto import tqdm

from config import CFG, set_seed
from data.dataset import (LABELS, NEW_DATASET_PATH, class_weights, compact_statement, load_problems, report,
                          split_indices)
from tracking import log_classification, log_image, log_table, start_run

SEED = set_seed()                # CFG.seed from config.py

# --- settings (model-specific, so they live here and not in config.py) -----------------------
TRAIN_LAYERS = 4       # top encoder layers that are trained (0 = only the decision head)
EPOCHS = 6             # at most; early stopping usually ends sooner
PATIENCE = 2           # stop when validation macro-F1 hasn't improved for this many epochs
BATCH_SIZE = 8
GRAD_ACCUM = 2         # gradients of 2 batches per update → effective batch 16
LR_ENCODER = 2e-5      # small: the encoder is already good, only nudge it
LR_HEAD = 1e-4         # larger: the head has to learn a new question
WEIGHT_DECAY = 0.01
WARMUP = 0.1           # share of updates with a rising learning rate
RETRAIN = False        # True = train even if a checkpoint with the same settings exists

SETTINGS = {"laya_model": CFG.laya_model, "train_layers": TRAIN_LAYERS, "epochs": EPOCHS, "patience": PATIENCE,
            "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM, "lr_encoder": LR_ENCODER, "lr_head": LR_HEAD,
            "weight_decay": WEIGHT_DECAY, "warmup": WARMUP, "seed": SEED, "sample_limit": CFG.sample_limit,
            "class_weighting": CFG.class_weighting}
CKPT = CFG.models_dir / "laya_finetuned.pt"
OUT = CFG.results_dir / "laya"; OUT.mkdir(parents=True, exist_ok=True)
FIG = CFG.figures_dir

## 1. Data

The compact statement (statement + constraints, no worked examples), exactly as in the zero-shot run, so
that the "before training" row below reproduces the zero-shot result.

In [ ]:
slugs, texts, y = load_problems()
states = [compact_statement(t) for t in texts]
tr_idx, va_idx, te_idx = split_indices(y, with_val=True)      # the shared test set
y_tr, y_va, y_te = y[tr_idx], y[va_idx], y[te_idx]
print(f"train {len(tr_idx)}, validation {len(va_idx)}, test {len(te_idx)}")

## 2. Model and inputs

`laya.load()` gives an `Agent`; the PyTorch model inside is `agent.model`. We build the input sequences
with Laya's own `build_sequence`, so the format is exactly what the model was trained on.

In [ ]:
agent = laya.load(CFG.laya_model)
tok, model, device = agent.tok, agent.model, agent.device
MAX_LEN, HEAD_MAX_LEN = agent.cfg.get("max_len", 512), agent.cfg.get("head_max_len", 192)
print("device:", device)

DIFF_OPTS = {   # the same options as the zero-shot `difficulty` question
    "easy":   "direct implementation or one basic idea such as a loop, hash map or two pointers",
    "medium": "a standard algorithm or data structure (BFS/DFS, binary search, heap, basic dynamic programming) applied with care",
    "hard":   "several ideas combined or an advanced technique (segment tree, hard dynamic programming, advanced graph theory) under tight constraints",
}
QUESTION = {"t": "choice", "ins": "How difficult is this LeetCode programming problem?", "crit": DIFF_OPTS}
KEYS = ("input_ids", "attention_mask", "marker_pos", "marker_mask", "qtype")


def encode(state):
    ids, markers = build_sequence(tok, state, QUESTION, MAX_LEN, HEAD_MAX_LEN)
    assert len(markers) == 3, "an option marker was cut off"
    return {"ids": ids, "markers": markers, "qtype": QTYPES["choice"]}


def to_device(item_list):
    b = collate_items([item_list], tok.pad_token_id)
    return [b[k].to(device) for k in KEYS]


@torch.no_grad()
def predict_proba(item_list, batch_size=2 * BATCH_SIZE):
    model.eval()
    out = []
    for i in range(0, len(item_list), batch_size):
        logits, _ = model(*to_device(item_list[i:i + batch_size]))
        out.append(torch.softmax(logits[:, :3].float(), -1).cpu().numpy())
    return np.concatenate(out)


items = [encode(s) for s in tqdm(states, desc="tokenising")]
lengths = np.array([len(it["ids"]) for it in items])
print(f"sequence length: median {int(np.median(lengths))}, max {lengths.max()} (limit {MAX_LEN})")

## 3. What gets trained

Everything is frozen first, then the top `TRAIN_LAYERS` encoder layers and the decision head are
unfrozen. The `act_head` (Laya's "should I act or escalate" output) isn't used here and stays frozen.

In [ ]:
n_layers = len(model.encoder.layers)
k = min(TRAIN_LAYERS, n_layers)
for p in model.parameters():
    p.requires_grad = False
encoder_top = list(model.encoder.layers[n_layers - k:]) + ([model.encoder.final_norm] if k else [])
head = [m for m in (model.head, model.type_emb, model.scorer) if m is not None]
for m in encoder_top + head:
    for p in m.parameters():
        p.requires_grad = True

trainable = {n: p for n, p in model.named_parameters() if p.requires_grad}
n_all = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in trainable.values())
print(f"encoder layers: {n_layers}, trained: top {k}")
print(f"parameters: {n_all / 1e6:.0f} M in total, {n_train / 1e6:.1f} M trained ({n_train / n_all:.0%})")

## 4. Before training (sanity check)

The untouched model on the validation and test problems. Macro-F1 on test should match the zero-shot
`choice` row in `laya_experiments.ipynb` (≈ 0.30): same model, same input, same question.

In [ ]:
P_va0, P_te0 = predict_proba([items[i] for i in va_idx]), predict_proba([items[i] for i in te_idx])
before = {"val": report(y_va, P_va0.argmax(1), name="before training (validation)"),
          "test": report(y_te, P_te0.argmax(1), name="before training (test)")}

## 5. Train (or load)

* Loss: cross-entropy with class weights (`CFG.class_weighting`), so Easy and Hard count as much as Medium.
* AdamW, two learning rates (encoder small, head larger), warm-up then linear decay, gradient clipping at 1.
* After each epoch: validation macro-F1. The best epoch's weights are kept and saved; training stops
  after `PATIENCE` epochs without improvement.

Only the trained weights are saved (~0.3 GB, not the whole 1.6 GB model).

In [ ]:
def load_checkpoint():
    if RETRAIN or not CKPT.exists():
        return None
    ck = torch.load(CKPT, map_location="cpu")
    if ck.get("settings") != SETTINGS:
        changed = {k: (ck.get("settings", {}).get(k), v) for k, v in SETTINGS.items()
                   if ck.get("settings", {}).get(k) != v}
        print(f"checkpoint was trained with other settings {changed}: training again")
        return None
    return ck


def train():
    w = class_weights(y_tr)
    loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor([w[c] for c in range(3)], dtype=torch.float32,
                                                            device=device))
    enc_params = [p for m in encoder_top for p in m.parameters()]
    head_params = [p for m in head for p in m.parameters()]
    groups = [{"params": head_params, "lr": LR_HEAD}] + ([{"params": enc_params, "lr": LR_ENCODER}] if enc_params else [])
    opt = torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)
    updates = EPOCHS * math.ceil(math.ceil(len(tr_idx) / BATCH_SIZE) / GRAD_ACCUM)
    warm = max(1, int(WARMUP * updates))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warm, max(0.0, (updates - s) / max(1, updates - warm))))

    rng = np.random.default_rng(SEED)
    history, best, best_state, bad = [], -1.0, None, 0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        t0, losses = time.time(), []
        order = rng.permutation(tr_idx)
        batches = [order[i:i + BATCH_SIZE] for i in range(0, len(order), BATCH_SIZE)]
        opt.zero_grad()
        for step, part in enumerate(tqdm(batches, desc=f"epoch {epoch}", leave=False), 1):
            logits, _ = model(*to_device([items[i] for i in part]))
            loss = loss_fn(logits[:, :3].float(), torch.tensor(y[part], device=device))
            (loss / GRAD_ACCUM).backward()
            losses.append(loss.item())
            if step % GRAD_ACCUM == 0 or step == len(batches):
                torch.nn.utils.clip_grad_norm_(trainable.values(), 1.0)
                opt.step(); sched.step(); opt.zero_grad()

        P_va = predict_proba([items[i] for i in va_idx])
        row = {"epoch": epoch, "train_loss": float(np.mean(losses)),
               "val_loss": float(log_loss(y_va, P_va, labels=[0, 1, 2])),
               "val_macro_f1": float(f1_score(y_va, P_va.argmax(1), average="macro")),
               "minutes": (time.time() - t0) / 60}
        history.append(row)
        run.log({f"epoch/{k}": v for k, v in row.items() if k != "epoch"}, step=epoch)
        print(f"epoch {epoch}: train loss {row['train_loss']:.3f}, val loss {row['val_loss']:.3f}, "
              f"val macro-F1 {row['val_macro_f1']:.3f} ({row['minutes']:.1f} min)")
        if row["val_macro_f1"] > best + 1e-4:
            best, bad = row["val_macro_f1"], 0
            best_state = {n: p.detach().cpu().clone() for n, p in trainable.items()}
            torch.save({"settings": SETTINGS, "trainable": best_state, "history": history, "best_epoch": epoch}, CKPT)
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f"early stop: no improvement for {PATIENCE} epochs")
                break
    # the saved checkpoint holds the history up to the best epoch; store the full one
    ck = torch.load(CKPT, map_location="cpu")
    ck["history"] = history
    torch.save(ck, CKPT)
    return ck


ck = load_checkpoint()
trained_now = ck is None
run = start_run("laya", "finetune", f"top{TRAIN_LAYERS}", job_type="train" if trained_now else "eval", config=SETTINGS)
if trained_now:
    ck = train()
else:
    print(f"loaded {CKPT.name} (best epoch {ck['best_epoch']}); set RETRAIN = True to train again")
missing = set(trainable) - set(ck["trainable"])
assert not missing, f"checkpoint is missing {len(missing)} tensors, e.g. {sorted(missing)[:3]}"
model.load_state_dict(ck["trainable"], strict=False)       # the best epoch's weights
history = pd.DataFrame(ck["history"])
history

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(history.epoch, history.train_loss, "o-", label="train")
axes[0].plot(history.epoch, history.val_loss, "o-", label="validation")
axes[0].set_title("loss (weighted cross-entropy)"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(history.epoch, history.val_macro_f1, "o-", color="#16a085")
axes[1].axvline(ck["best_epoch"], color="k", ls=":", lw=1)
axes[1].set_title("validation macro-F1 (dotted = kept epoch)"); axes[1].set_xlabel("epoch")
fig.tight_layout()
fig.savefig(FIG / "laya_finetune_curves.png", dpi=CFG.fig_dpi)
plt.show()

## 6. Calibrate and tune on validation, test once

Three versions of the fine-tuned model, **all chosen on the validation set**:

1. **raw**: argmax of the fine-tuned probabilities;
2. **calibrated**: temperature + class bias fitted on validation (fixes over- / under-confidence; argmax can move a bit because of the bias);
3. **F1-tuned**: calibrated + an extra Easy/Hard shift that maximises validation macro-F1.

The version with the best validation macro-F1 is *the* fine-tuned result; the test numbers of the others
are shown only for insight.

In [ ]:
def fit_temp_bias(P, y_true):
    # new logits = log p / T + b  (b_easy = 0), fitted by minimising log-loss
    L = np.log(np.clip(P, 1e-9, 1))
    def nll(w):
        T, b = np.exp(w[0]), np.r_[0.0, w[1:]]
        z = L / T + b; z -= z.max(1, keepdims=True)
        logp = z - np.log(np.exp(z).sum(1, keepdims=True))
        return -logp[np.arange(len(y_true)), y_true].mean()
    w = minimize(nll, np.zeros(3), method="Nelder-Mead").x
    T, b = np.exp(w[0]), np.r_[0.0, w[1:]]
    def apply(P2):
        z = np.log(np.clip(P2, 1e-9, 1)) / T + b; z -= z.max(1, keepdims=True)
        e = np.exp(z); return e / e.sum(1, keepdims=True)
    return apply, T, b


def tune_shift(L, y_true, grid=np.linspace(-2, 3, 51)):
    # extra log-bias for Easy and Hard (Medium stays 0) with the best macro-F1
    best_f1, best = -1.0, np.zeros(3)
    for b_easy in grid:
        for b_hard in grid:
            s = np.array([b_easy, 0.0, b_hard])
            f1 = f1_score(y_true, (L + s).argmax(1), average="macro")
            if f1 > best_f1 + 1e-9:
                best_f1, best = f1, s
    return best, best_f1


log = lambda P: np.log(np.clip(P, 1e-9, 1))
P_va, P_te = predict_proba([items[i] for i in va_idx]), predict_proba([items[i] for i in te_idx])
cal, T, b = fit_temp_bias(P_va, y_va)
shift, _ = tune_shift(log(cal(P_va)), y_va)
print(f"calibration: T = {T:.2f}, bias = {b.round(2)}; F1 shift (Easy, Medium, Hard) = {shift.round(2)}")

VARIANTS = {"raw": lambda P: log(P),
            "calibrated": lambda P: log(cal(P)),
            "F1-tuned": lambda P: log(cal(P)) + shift}
rows = []
for name, f in VARIANTS.items():
    m_va = report(y_va, f(P_va).argmax(1), name=f"fine-tuned {name} (validation)")
    m_te = report(y_te, f(P_te).argmax(1), name=f"fine-tuned {name} (test)")
    rows.append({"variant": name, **{f"val_{k}": v for k, v in m_va.items()}, **{f"test_{k}": v for k, v in m_te.items()}})
table = pd.DataFrame(rows).set_index("variant")
table.loc["before training"] = pd.Series({**{f"val_{k}": v for k, v in before["val"].items()},
                                          **{f"test_{k}": v for k, v in before["test"].items()}})
best_variant = table.drop("before training")["val_macro_f1"].idxmax()     # chosen on validation
print("\nchosen by validation macro-F1:", best_variant)
table.to_csv(OUT / "finetune_results.csv")
table.style.format(precision=3)

## 7. New problems (published after the original download)

The same check as Extra 2 in `laya_experiments.ipynb` and Extra 3 in the Llama notebook: the fine-tuned
model, unchanged, on the 876 problems with ID ≥ 3000. ⚠ They have more Hard problems (~30% vs ~22%), so
compare macro-F1 / QWK, not accuracy.

In [ ]:
pred_te = VARIANTS[best_variant](P_te).argmax(1)
if NEW_DATASET_PATH.exists():
    new_slugs, new_texts, y_new = load_problems(path=NEW_DATASET_PATH)
    new_items = [encode(compact_statement(t)) for t in tqdm(new_texts, desc="tokenising new problems")]
    P_new = predict_proba(new_items)
    pred_new = VARIANTS[best_variant](P_new).argmax(1)
    m_new = report(y_new, pred_new, name=f"fine-tuned {best_variant} (new problems)")
else:
    y_new = pred_new = None
    print("no new problems: run `python data/fetch_new_problems.py` to add this check")

panels = [("test split", y_te, pred_te)] + ([("new problems", y_new, pred_new)] if y_new is not None else [])
fig, axes = plt.subplots(1, len(panels), figsize=(4.4 * len(panels), 4), squeeze=False)
for ax, (title, yt, yp) in zip(axes[0], panels):
    cm = confusion_matrix(yt, yp, labels=[0, 1, 2])          # rows = true, columns = predicted
    sns.heatmap(cm, annot=True, fmt="d", cmap="BuGn", xticklabels=LABELS, yticklabels=LABELS, cbar=False, ax=ax)
    ax.set_title(f"fine-tuned {best_variant}, {title}\n(macro-F1 {f1_score(yt, yp, average='macro'):.2f})", fontsize=10)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
fig.tight_layout()
fig.savefig(FIG / "laya_finetuned.png", dpi=CFG.fig_dpi)
plt.show()

In [ ]:
# MLflow: "laya-finetune-top<k>-s<seed>" in experiment ".../laya" (opened in §5, so the epochs are in it too)
run.config.update({"best_variant": best_variant, "best_epoch": ck["best_epoch"],
                   "temperature": round(float(T), 3), "bias": b.round(3).tolist(), "f1_shift": shift.round(2).tolist()})
log_classification(run, y_te, pred_te, prefix=f"test/{best_variant}")
if y_new is not None:
    log_classification(run, y_new, pred_new, prefix=f"new/{best_variant}")
log_table(run, "variants", table.reset_index())
log_table(run, "history", history)
for f in ["laya_finetune_curves", "laya_finetuned"]:
    log_image(run, f"figures/{f}", FIG / f"{f}.png")
run.finish()

## Where this sits

Put the chosen variant's test macro-F1 next to the others (all on the same 474 test problems; ±0.05 is noise):

| Model | test macro-F1 |
|---|---|
| always Medium | 0.225 |
| TF-IDF + LinearSVC | ≈ 0.48 |
| Laya embeddings + signals + LR (frozen Laya) | 0.494 |
| RNN (BiGRU) | 0.509 |
| **Laya fine-tuned** | see §6 |
| Llama 3.1 8B, 6-shot calibrated (partly memory) | 0.57 |

**Knobs worth trying if you want more** (change a setting and re-run; a changed setting retrains
automatically): `TRAIN_LAYERS` 0 / 8 / 12 (more layers = more capacity, slower, more overfitting),
`LR_ENCODER` 1e-5 / 5e-5, `EPOCHS`. Each run gets its own MLflow entry (`laya-finetune-top<k>-s42`).